In [158]:
import numpy as np
import pandas as pd
import pandapower as pp
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pickle
import pp_heig_simulation as pp_sim
import pp_heig_plot as pp_plot
from datetime import time
import re

In [159]:
## Import pickle
net_trey = pp.from_pickle("input-data/trey_net_student.p")

In [ ]:
### Line parameters
## Add a new column "length_km" to net_trey
net_trey.line.insert(
    7, "length_km", [0.180, 0.100, 0.115, 0.080, 0.090, 0.135, 0.100, 0.205, 0.085, 0.255, 0.340]
) 

## Create new column "cable_type" by removing the unwanted value (_x) use to differentiate each line from "name" with regex
# re.sub replaces the matches with the given substitute string
# First argument: regex pattern
# Second argument: substitute string
# Third argument: input string
net_trey.line["cable_type"] = net_trey.line.apply(
    lambda x: re.sub(r"_[0-9]$", "", x["name"]), axis=1
)
 
## Create parameter table for cable types
cable_types = pd.DataFrame(
    {
        "cable_type": ["GKN3x150_150", "GKN3X95_95", "GKN3X50_50", "GKT3X50_50","GKN3X240_240"],
        "r_ohm_per_km"  : [0.124, 0.193, 0.387, 0.387, 0.0754],
        "x_ohm_per_km"  : [0.07, 0.07, 0.07, 0.07, 0.07],
        "c_uf_per_km"   : [0.349, 0.338, 0.298, 0.298, 0.346],
        "max_i_ka"      : [0.400, 0.252, 0.170, 0.170, 0.512],
    }
)
 
## Merge net_trey.line with cable_types on "cable_type"
net_trey.line = net_trey.line.merge(cable_types, on="cable_type", how="left")

## Show updated net_trey.line
net_trey.line

,name,from_bus,to_bus,g_us_per_km,df,std_type,in_service,length_km,cable_type,r_ohm_per_km,x_ohm_per_km,c_uf_per_km,max_i_ka
0,GKN3x150_150,0,1,0.0,1.0,None,True,0.180,GKN3x150_150,0.1240,0.07,0.349,0.400
1,GKN3x150_150_2,1,2,0.0,1.0,None,True,0.100,GKN3x150_150,0.1240,0.07,0.349,0.400
2,GKN3X95_95,1,3,0.0,1.0,None,True,0.115,GKN3X95_95,0.1930,0.07,0.338,0.252
3,GKT3X50_50,3,4,0.0,1.0,None,True,0.080,GKT3X50_50,0.3870,0.07,0.298,0.170
4,GKT3X50_50_2,4,5,0.0,1.0,None,True,0.090,GKT3X50_50,0.3870,0.07,0.298,0.170
5,GKN3x150_150_3,0,6,0.0,1.0,None,True,0.135,GKN3x150_150,0.1240,0.07,0.349,0.400
6,GKN3X95_95_2,6,7,0.0,1.0,None,True,0.100,GKN3X95_95,0.1930,0.07,0.338,0.252
7,GKN3x150_150_4,7,8,0.0,1.0,None,True,0.205,GKN3x150_150,0.1240,0.07,0.349,0.400
8,GKN3X50_50,7,9,0.0,1.0,None,True,0.085,GKN3X50_50,0.3870,0.07,0.298,0.170
9,GKN3X50_50_2,6,10,0.0,1.0,None,True,0.255,GKN3X50_50,0.3870,0.07,0.298,0.170


In [ ]:
## Buses parameters
net_trey.bus["vn_kv"] = net_trey.bus.apply(
    lambda row: 18.3 if row["type"] == "Slack" else (0.420 if row["type"] == "PQ" else None),
    axis=1,
)

net_trey.bus

,name,type,zone,in_service,vn_kv
0,STMT003438,PQ,Trafo,True,0.42
1,CDBT004764,PQ,North,True,0.42
2,CDBT003746,PQ,North,True,0.42
3,CDBT004760,PQ,North,True,0.42
4,CDBT012139,PQ,North,True,0.42
5,CDBT900784,PQ,North,True,0.42
6,CDBT901452,PQ,South,True,0.42
7,CDBT004774,PQ,South,True,0.42
8,CDBT901604,PQ,South,True,0.42
9,CDBT016055,PQ,South,True,0.42


In [ ]:
## Connect external grid to bus 12
net_trey.ext_grid.insert(2, "bus", 12)

In [ ]:
## Plot grid
pp_plot.plot_power_network(
    net=net_trey,
    plot_title="Trey",
    filename="trey_grid_example",
)